# LangGraph

En este laboratorio, aprenderás a crear agentes de IA inteligentes utilizando LangGraph, un potente framework de Python para la creación de flujos de trabajo basados ​​en estados. Comenzarás comprendiendo los componentes principales de LangGraph y cómo interactúan para crear sistemas estructurados de toma de decisiones. Aprenderás a definir tareas como nodos, conectarlas mediante una estructura de grafo y crear agentes capaces de razonar, planificar y actuar en función de las entradas. Mediante ejemplos prácticos, desarrollarás dos aplicaciones: un flujo de trabajo de autenticación que valida las credenciales de usuario con un manejo de errores adecuado y un sistema de preguntas y respuestas que responde a consultas sobre LangGraph. Al finalizar este laboratorio, podrás crear tus propios agentes de IA que procesen información, tomen decisiones y se adapten a condiciones cambiantes utilizando flujos de trabajo tanto lineales como cíclicos.

## 1. ¿Qué es LangGraph?

LangGraph es un potente framework diseñado para ayudar a los desarrolladores a crear agentes de IA inteligentes capaces de pensar, planificar y actuar en tu nombre. En esencia, LangGraph te permite desglosar tareas complejas en pasos más pequeños y manejables, organizarlos lógicamente y conectarlos como un mapa de ruta mediante **grafos**. Estos grafos representan visualmente el flujo de tareas, facilitando la comprensión de cómo se conecta cada paso y en qué orden deben ejecutarse.

Piensa en LangGraph como un conjunto de herramientas para crear asistentes inteligentes. Estos asistentes no solo siguen instrucciones, sino que comprenden lo que quieres, encuentran la mejor manera de lograrlo y ejecutan las tareas automáticamente. Con LangGraph, puedes diseñar sistemas de IA estructurados, adaptables y capaces de gestionar desde flujos de trabajo sencillos hasta tareas complejas de resolución de problemas.

## 2. ¿Por Qué LangGraph?

LangGraph ofrece ventajas únicas para la creación de agentes de IA inteligentes, especialmente en comparación con frameworks como LangChain. Una de sus características más destacadas es la capacidad de crear **grafos cíclicos**, además de grafos acíclicos dirigidos (DAG). He aquí por qué esto es importante:

**1. Compatibilidad con Grafos Cíclicos**

Si bien herramientas como LangChain se limitan a los DAG —donde las tareas deben seguir un flujo lineal estricto sin bucles—, LangGraph permite **grafos cíclicos**, donde las tareas pueden retomar pasos anteriores. Esto es crucial para flujos de trabajo que requieren procesos iterativos, bucles de retroalimentación o reintentos.

Por ejemplo:

- Un agente de IA que analiza los comentarios de los clientes podría retomar pasos anteriores para refinar su comprensión con base en nuevos datos.

- Un proceso para refinar un documento podría repetir los pasos de borrador, revisión y modificación varias veces.

**3. Mayor Flexibilidad en el Flujo de Trabajo**

Los grafos cíclicos hacen de LangGraph la solución ideal para escenarios reales donde las tareas no siempre siguen un camino unidireccional. Puede gestionar flujos de trabajo dinámicos donde los agentes necesitan adaptarse, reevaluar o repetir pasos según sea necesario, lo que proporciona una mayor flexibilidad.

**4. Potentes Procesos Iterativos**

Al habilitar bucles, LangGraph admite procesos como la optimización, el refinamiento de datos o los ajustes en tiempo real, lo que garantiza que el agente de IA ofrezca resultados más precisos y significativos.

**5. Solución Integral**

LangGraph no solo gestiona tareas, sino que también integra las ventajas de los DAG (para flujos estructurados) y los grafos cíclicos (para flujos de trabajo iterativos). Esto lo convierte en un marco más versátil para el diseño de agentes de IA inteligentes y adaptables.

Con LangGraph, no estará limitado por flujos de tareas unidireccionales. Su compatibilidad con gráficos cíclicos permite modelar con facilidad problemas más complejos del mundo real, lo que la convierte en una opción ideal para el desarrollo de IA.

## 3. Configuración


Para este laboratorio, utilizaremos las siguientes bibliotecas:

* [`langgraph==0.2.57`](https://pypi.org/project/langgraph/) — para construir flujos de trabajo de IA reactivos, con estado y de múltiples pasos.

* [`langchain-ibm==0.3.10`](https://pypi.org/project/langchain-ibm/) — para integrar servicios de IBM (como Watsonx) con las herramientas de LangChain.

### *3.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *3.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [1]:
from langgraph.graph import StateGraph

### *3.3. Aviso Legal Sobre la API*

Este laboratorio utiliza módulos LLM proporcionados por OpenAI. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente fuera** del entorno JupyterLab de Skills Network, deberá configurar sus propias claves API. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

Si ejecuta este laboratorio localmente, deberá configurar sus propias claves API. Este laboratorio utiliza los módulos `ChatOpenAI` y `ChatWatsonx` de `langchain`. La siguiente sección muestra ambas configuraciones con instrucciones. **Reemplace todas las instancias** de ambos módulos con los módulos completos que se muestran a continuación en todo el laboratorio.

<p style='color: red'><b>NO ejecute la siguiente celda si no está ejecutando el laboratorio localmente, ya que provocará errores.</b>

In [2]:
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(
    model= "gpt-4o-mini",
    temperature= 0.4,
    max_completion_tokens= 2048
)

## 4. Componentes de LangGraph


La principal fortaleza de LangGraph reside en su diseño estructurado e intuitivo, compuesto por bloques de construcción esenciales que permiten la creación de agentes de IA inteligentes, con etapas predeterminadas que indican el inicio y el final del flujo de trabajo.

En este ejemplo, definimos una estructura de estado para un **flujo de trabajo de autenticación de usuario**. El estado, llamado `EstadoAutenticacion`, es un diccionario tipado que contiene información sobre las credenciales de un usuario (nombre de usuario y contraseña) y su estado de autenticación (`es_autenticado`), como se muestra aquí:

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/dhl478pwbZzrxzmcjJY4xw/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM.png" alt="Screenshot" width="300">
</div>


### *4.1. Estados*


Los estados representan la **condición o contexto actual** dentro de un flujo de trabajo. Almacenan y gestionan información a medida que el agente avanza de un nodo al siguiente.

Por ejemplo:
- Un estado puede capturar la entrada del usuario, almacenar los resultados de una consulta a la base de datos o reflejar el estado de un proceso en curso.

Los estados garantizan que el agente de IA tenga acceso a la información relevante en el momento adecuado, lo que permite un comportamiento dinámico y sensible al contexto.

#### 4.1.1. Ejemplo: Descripción de la clase - `AuthState`

Mediante `from typing import TypedDict, Optional`, la clase `EstadoAutenticacion` se define como un `TypedDict` que especifica la estructura de un diccionario que representa el estado de autenticación de un usuario. Cada clave tiene un tipo específico y todos los campos son opcionales (`Optional`), lo que significa que pueden contener un valor del tipo especificado o ser `None`.

#### 4.1.2. Claves y tipos de estado

- **`username`**: `Optional[str]` - El nombre de usuario; Puede ser una cadena de texto o `None`.

- **`password`**: `Optional[str]` - La contraseña del usuario; puede ser una cadena de texto o `None`.

- **`es_autenticado`**: `Optional[bool]` - Indica si el usuario está autenticado; puede ser un valor booleano o `None`.

- **`salida`**: `Optional[str]` - Un mensaje o resultado relacionado con la autenticación; puede ser una cadena de texto o `None`.

Esta estructura garantiza que el estado de autenticación esté definido de forma consistente y sea seguro en cuanto a tipos, a la vez que contempla escenarios en los que algunos campos pueden no estar disponibles o no utilizarse.

In [3]:
from typing import TypedDict, Optional

class EstadoAutenticacion(TypedDict):
    username: Optional[str] 
    password: Optional[str]
    es_autenticado: Optional[bool]
    salida: Optional[str]

#### 4.1.3. Ejemplos de objetos y sus estados

##### *4.1.3.1. Objeto 1: Inicio de sesión exitoso*

Aquí tienes un ejemplo del objeto ```EstadoAutenticacion``` con un inicio de sesión exitoso:

In [4]:
estado_autenticacion_1: EstadoAutenticacion = {
    "username": "alice123",
    "password": "123",
    "es_autenticado": True,
    "salida": "Inicio de sesión exitoso"
}
print(f"estado_autenticacion_1: {estado_autenticacion_1}")

estado_autenticacion_1: {'username': 'alice123', 'password': '123', 'es_autenticado': True, 'salida': 'Inicio de sesión exitoso'}


##### *4.1.3.2. Objeto 2: Inicio de sesión no exitoso*

Aquí tienes un ejemplo del objeto ```EstadoAutenticacion``` con un inicio de sesión no exitoso:


In [5]:
estado_autenticacion_2: EstadoAutenticacion = {
    "username":"",
    "password": "wrongpassword",
    "es_autenticado": False,
    "salida": "Inicio de sesión fallido: credenciales incorrectas"
}
print(f"estado_autenticacion_2: {estado_autenticacion_2}")

estado_autenticacion_2: {'username': '', 'password': 'wrongpassword', 'es_autenticado': False, 'salida': 'Inicio de sesión fallido: credenciales incorrectas'}


Este estado sirve como base para los flujos de trabajo que implican la autenticación de usuarios. Se transmitirá entre los nodos de un grafo para validar las credenciales y actualizar el campo `es_autenticado` según corresponda.

### *4.2. Nodos*

#### 4.2.1. Definición del nodo de entrada

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QPsWYzaKkacLUn8V1SbyWA/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh.png" alt="Screenshot" width="300">
</div>


Ahora definimos el nodo `nodo_entrada`, que recopila el nombre de usuario y la contraseña si aún no se han proporcionado en el estado. Este nodo garantiza que el estado se complete con la información necesaria para la autenticación.

Este nodo suele ser el punto de partida en el grafo, asegurando que se recopilen los datos de entrada necesarios antes de proceder al paso de autenticación.

In [6]:
def nodo_entrada(estado):
    print(estado)

    if estado.get('username', "") == "":
        username = input("Ingrese su nombre de usuario: ")

    password = input("Ingrese su contraseña: ")
    
    if estado.get('username', "") == "":
        return {"username": username, "password": password}
    else:
        return {"password": password}

Le pasamos el primer objeto con todos los campos, y ```nodo_entrada``` nos pide la contraseña:

In [7]:
nodo_entrada(estado_autenticacion_1)

{'username': 'alice123', 'password': '123', 'es_autenticado': True, 'salida': 'Inicio de sesión exitoso'}


{'password': '123'}

In [8]:
nodo_entrada(estado_autenticacion_2)

{'username': '', 'password': 'wrongpassword', 'es_autenticado': False, 'salida': 'Inicio de sesión fallido: credenciales incorrectas'}


{'username': 'martin', 'password': '123'}

En este ejemplo, como no se ha proporcionado ningún nombre de usuario, la función nos pide que introduzcamos uno.

#### 4.2.2. Definición del nodo Validar credenciales

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/OI9x1j0MXYtlsjdGGPemqA/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-1-.png" alt="Screenshot" width="300">
</div>

El nodo es un componente fundamental de un grafo que encapsula una unidad de computación o funcionalidad. Representa un paso en un flujo de trabajo o proceso, que generalmente consiste en recibir una entrada, realizar una acción y proporcionar una salida. Cada nodo se conecta con otros para definir el flujo de lógica o datos.

En su ejemplo, el nodo `nodo_validar_credenciales` funcionaría como un nodo en Langraph, realizando la tarea de validar las credenciales del usuario.

El nodo `nodo_validar_credenciales` recibe el estado actual como entrada y se encarga de verificar las credenciales del usuario (nombre de usuario y contraseña) proporcionadas en dicho estado. Según el resultado de la validación, actualiza el estado con un valor `es_autenticado`, que indica si la autenticación fue exitosa o no. Este nodo garantiza que el resultado de la validación se agregue al estado, lo que permite al grafo determinar el siguiente paso en el flujo de trabajo en función de si la autenticación fue exitosa.

In [9]:
def nodo_validar_credenciales(state):
    # Extraer las credenciales del estado.
    username = state.get("username", "")
    password = state.get("password", "")

    print("Username :", username, "Password :", password)

    # Simular la validación de credenciales (en un caso real, esto implicaría verificar contra una base de datos o servicio de autenticación).
    if username == "test_user" and password == "secure_password":
        es_autenticado = True
    else:
        es_autenticado = False

    # Actualizar el estado con el resultado de la autenticación.
    return {"es_autenticado": es_autenticado}

Podemos aplicar `nodo_validar_credenciales` a dos objetos: uno con formato incorrecto y otro con formato correcto, para probar su funcionalidad.

##### *4.2.2.1. Formato incorrecto*

In [10]:
nodo_validar_credenciales(estado_autenticacion_1)

Username : alice123 Password : 123


{'es_autenticado': False}

##### *4.2.2.2. Formato correcto*


In [11]:
estado_autenticacion_3: EstadoAutenticacion = {
    "username":"test_user",
    "password":  "secure_password",
    "es_autenticado": False,
    "salida": "Error de autenticación: credenciales incorrectas. Intente nuevamente."
}
print(f"estado_autenticacion_3: {estado_autenticacion_3}")

estado_autenticacion_3: {'username': 'test_user', 'password': 'secure_password', 'es_autenticado': False, 'salida': 'Error de autenticación: credenciales incorrectas. Intente nuevamente.'}


In [12]:
nodo_validar_credenciales(estado_autenticacion_3)

Username : test_user Password : secure_password


{'es_autenticado': True}

#### 4.2.3. Definición del nodo de éxito

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/PwwDttqbSAwfrYvOQP2X-A/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-2-.png" alt="Screenshot" width="300">
</div>


El nodo `nodo_exito` recibe como entrada el estado actual y se activa cuando el proceso de autenticación es exitoso. Devuelve un mensaje de éxito para indicar que el usuario ha sido autenticado.

In [ ]:
# Definir el nodo de éxito que se activa cuando la autenticación es exitosa.
def nodo_exito(estado):
    return {"salida": "Autenticación exitosa. Bienvenido!"}

In [15]:
nodo_exito(estado_autenticacion_3)

{'salida': 'Autenticación exitosa. Bienvenido!'}

Este nodo está conectado en el grafo para gestionar el flujo cuando las credenciales del usuario se validan correctamente. Proporciona un resultado positivo para el flujo de trabajo.

#### 4.2.4. Definición del nodo de fallo

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/69hJU9b_hk_UNARWJW6IEw/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-3-.png" alt="Screenshot" width="300">
</div>



El nodo `nodo_fallo` recibe como entrada el estado actual y se activa cuando falla el proceso de autenticación. Devuelve una salida que indica el fallo de autenticación.

In [16]:
# Definir el nodo de fallo que se activa cuando la autenticación falla.
def nodo_fallo(estado):
    return {"salida": "Autenticación fallida. Por favor, inténtelo de nuevo!"}

Este nodo está conectado en el gráfico para gestionar el flujo cuando las credenciales del usuario no superan la validación, proporcionando información sobre el intento de autenticación fallido.

#### 4.2.5. Definición del nodo del enrutador

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/FpW7oG5oTxIPsaOqpxT7ow/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-4-.png" alt="Screenshot" width="300">
</div>


El nodo `enrutador` actúa como un punto de toma de decisiones en el flujo de trabajo. Recibe el estado actual como entrada y determina el siguiente nodo a ejecutar en función del valor `es_autenticado` del estado.

In [30]:
def enrutador(estado):
    if estado['es_autenticado']:
        return "nodo-exito"
    else:
        return "nodo-fallo"

Este nodo garantiza que el gráfico transite al nodo apropiado (ya sea el de éxito o el de fracaso) según si la autenticación fue exitosa o no. Es una parte esencial para gestionar la lógica condicional en el flujo de trabajo.

#### 4.2.6. Creación del grafo

Para comenzar a construir el flujo de trabajo, necesitamos crear un grafo que sirva de base para conectar los nodos y definir la lógica de la aplicación. Creamos una nueva instancia de `StateGraph` utilizando nuestra estructura `EstadoAutenticacion`, que actúa como un modelo para el estado de la aplicación. Este grafo gestionará el flujo de ejecución entre los nodos, asegurando un flujo de trabajo fluido y organizado.

In [53]:
from langgraph.graph import StateGraph
from langgraph.graph import END

# Crear la instancia del grafo utilizando la estructura de estado definida.
flujo_trabajo = StateGraph(EstadoAutenticacion)
flujo_trabajo

#### 4.2.7. Agregar nodos al grafo

Ahora, agregamos nodos al grafo para definir las tareas y la lógica del flujo de trabajo. Los nodos se agregan mediante el método `add_node`, que requiere dos argumentos:

1. **Nombre del nodo**: Un identificador único de cadena para el nodo.

2. **Función del nodo**: La función que ejecutará la lógica para este nodo.

Para recopilar la información del usuario para la autenticación, añadimos el nodo `nodo-entrada` al grafo mediante el método `add_node`. Este nodo solicita al usuario que introduzca su nombre de usuario y contraseña si aún no están presentes en el estado.

- `"nodo-entrada"`: Este es el identificador único del nodo de entrada.

- `nodo_entrada`: La función que recopila el nombre de usuario y la contraseña del usuario y actualiza el estado en consecuencia.

In [54]:
flujo_trabajo.add_node("nodo-entrada", nodo_entrada)

Para gestionar la lógica de autenticación, añadimos el nodo `nodo-validar-credenciales` al gráfico mediante el método `add_node`. Este nodo valida el nombre de usuario y la contraseña proporcionados por el usuario y actualiza el estado con el resultado de la autenticación.

In [55]:
flujo_trabajo.add_node("nodo-validar-credenciales", nodo_validar_credenciales)

Ahora, agregamos el nodo `nodo_exito` al gráfico usando el método `add_node`. Este nodo se activará si las credenciales se validan correctamente y devolverá un mensaje de éxito.

In [56]:
flujo_trabajo.add_node("nodo-exito", nodo_exito)

A continuación, añadimos el nodo `nodo_fallo` al gráfico mediante el método `add_node`. Este nodo se activará si las credenciales no son válidas y devolverá un mensaje de error.

In [57]:
flujo_trabajo.add_node("nodo-fallo", nodo_fallo)

### *4.3. Bordes*

Las aristas definen las conexiones entre nodos y representan el flujo de ejecución dentro del grafo. Determinan cómo el agente de IA transita de una tarea a otra según una lógica o condiciones predefinidas. En el flujo de trabajo de autenticación, las aristas guían el flujo de la aplicación, determinando la ruta a seguir en función de los resultados de la ejecución de cada nodo.

#### 4.3.1. Ejemplo de caso de uso de autenticación

- **Nodo de entrada**: El flujo se dirige desde este nodo al **Nodo de validación de credenciales**, donde se validan los datos introducidos por el usuario (nombre de usuario y contraseña).

- **Nodo de fallo**: Si la autenticación falla, el flujo regresa al **Nodo de entrada** para solicitar al usuario que vuelva a introducir sus credenciales.

- **Nodo de éxito**: Si la autenticación es exitosa, el flujo finaliza tras mostrar un mensaje de éxito que indica que el proceso de autenticación se ha completado correctamente.

#### 4.3.2. Añadiendo la arista entre nodo-entrada y nodo-validar-credenciales

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/2M4XoL8bc2Em8o_0T9XQbg/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-5-.png" alt="Screenshot" width="300">
</div>


Para establecer la conexión entre el nodo `nodo-entrada` y el nodo `nodo-validar-credenciales`, utilizamos el método `add_edge`. Esta arista representa el flujo desde la fase de entrada de datos del usuario hasta la fase de validación de credenciales, asegurando que, una vez que el usuario ingrese sus datos, el siguiente paso sea validarlos.

In [58]:
flujo_trabajo.add_edge("nodo-entrada", "nodo-validar-credenciales")

`add_edge(start, end)`: Este método crea una arista dirigida entre dos nodos, definiendo el flujo de ejecución de un nodo a otro.

- **`start`**: El nodo desde el que comienza el flujo. En este caso, es `"nodo-entrada"`, donde el usuario proporciona sus credenciales.

- **`end`**: El nodo al que finaliza el flujo. Aquí, es `"nodo-validar-credenciales"`, donde se validan las credenciales introducidas por el usuario.

#### 4.3.3. Agregar el borde entre el nodo-exito y el final


<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/8yOLvWWbyVbO8jdmSyEZGg/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-6-.png" alt="Screenshot" width="300">
</div>

Para definir el flujo de la aplicación tras una autenticación exitosa, creamos una arista entre el nodo **Éxito** y el nodo **FIN**. Esta arista indica la conclusión del proceso de autenticación, marcando la finalización exitosa de la tarea.

In [59]:
flujo_trabajo.add_edge("nodo-exito", END)

#### 4.3.4. Agregar la arista entre el nodo-falla y el nodo-entrada

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/zixPBpM-2GP1I0_X4wCloA/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-7-.png" alt="Screenshot" width="300">
</div>

En caso de que falle la autenticación, queremos permitir que el usuario vuelva a introducir sus credenciales. Para habilitar este flujo, creamos una conexión entre el **Nodo de Fallo** y el **Nodo de Entrada**. Esta conexión representa la transición de vuelta a la fase de entrada para que el usuario pueda intentar autenticarse de nuevo.

In [60]:
flujo_trabajo.add_edge("nodo-fallo", "nodo-entrada")

### *4.4. Aristas Condicionales*

Las aristas condicionales permiten la **toma de decisiones** al posibilitar transiciones entre nodos en función de condiciones específicas dentro del estado. Estas aristas definen el flujo de ejecución según resultados como la entrada del usuario, los resultados de la validación o cualquier otra lógica predefinida. Mediante el uso de aristas condicionales, el agente de IA puede elegir dinámicamente su ruta en función de los resultados de tareas anteriores.

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/dcXTJEOQ8nhYh7AvVdW9Mw/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-8-.png" alt="Screenshot" width="300">
</div>


## 5. Creación de un Flujo de Trabajo de Autenticación

### *5.1. Ejemplo de Caso de Uso de Autenticación*

En nuestro flujo de autenticación, podemos introducir aristas condicionales que determinen si el usuario se autenticó correctamente o no. Estas aristas se utilizan para decidir el siguiente paso en función del resultado de la autenticación.

- **Nodo de Validación de Credenciales**: Tras validar las credenciales del usuario, el sistema utiliza una arista condicional para decidir:

- Si `es_autenticado` es `True`, el flujo avanza al **Nodo de Éxito**.

- Si `es_autenticado` es `False`, el flujo regresa al **Nodo de Entrada** para que el usuario pueda intentar introducir sus credenciales de nuevo.

In [61]:
flujo_trabajo.add_conditional_edges("nodo-validar-credenciales", enrutador, {"nodo-exito": "nodo-exito", "nodo-fallo": "nodo-fallo"})

**`add_conditional_edges(start, router, conditions)`**: Este método define las transiciones condicionales desde un nodo dado.

- **`start`**: El nodo donde comienzan las aristas condicionales (en este caso, `"nodo-validacion-credenciales"`).


- **`router`**: Una función que determina la condición. Verifica el estado actual (como el estado `es_autenticado`) y devuelve el nodo apropiado al que se debe realizar la transición (ya sea `"nodo-exito"` o `"nodo-fallo"`).


- **`conditions`**: Un diccionario que asigna condiciones (como `"nodo-exito"` o `"nodo-fallo"`) a nodos de destino, indicando hacia dónde dirigir el flujo según la condición.

### *5.2. Estableciendo el Punto de Entrada*

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/rxNr7amJUiaEyhZKYc-JFg/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM%20copy-mh%20-9-.png" alt="Screenshot" width="300">
</div>

El **punto de entrada** define dónde comienza el flujo de trabajo. Al establecer un punto de entrada, se especifica el primer nodo que el agente de IA ejecutará al iniciarse el flujo de trabajo. En nuestro caso de uso de autenticación, queremos que el flujo de trabajo comience en el **Nodo de Entrada**, donde se le pedirá al usuario que ingrese sus credenciales.

Al definir el punto de entrada, nos aseguramos de que el flujo de trabajo se inicie en la fase de entrada, guiando al usuario paso a paso a través del proceso de autenticación.

In [62]:
flujo_trabajo.set_entry_point("nodo-entrada")

**`set_entry_point(node)`**: Este método establece el punto de inicio del flujo de trabajo.

- **`node`**: El nombre del nodo donde comenzará el flujo de trabajo. En este caso, `"nodo-entrada"`, lo que garantiza que el agente solicite al usuario sus credenciales antes de proceder con el proceso de autenticación.

### *5.3. Compilación del Flujo de Trabajo*

Una vez definidos todos los nodos y aristas del flujo de trabajo, el siguiente paso es compilarlo. La compilación transforma los nodos, aristas y la lógica definidos en una aplicación lista para usar que puede ejecutar las tareas definidas de forma secuencial o condicional, según la estructura del grafo.

In [63]:
app = flujo_trabajo.compile()

### *5.4. Ejecución de la Aplicación*

Una vez compilado el flujo de trabajo, podemos ejecutarlo invocando la aplicación con los datos de entrada necesarios. El método `invoke` recibe un estado inicial (un diccionario de valores de entrada) e inicia la ejecución desde el punto de entrada definido en el flujo de trabajo.

<p style='color: red'><b>Nota:</b> La contraseña correcta es <code>secure_password</code>, así que asegúrese de introducirla para autenticarse correctamente.</p>

In [64]:
entradas = {"username": "test_user"}
resultado = app.invoke(entradas)
print(resultado)

{'username': 'test_user'}
Username : test_user Password : secure_password
{'username': 'test_user', 'password': 'secure_password', 'es_autenticado': True, 'salida': 'Autenticación exitosa. Bienvenido!'}


Ahora estamos imprimiendo el resultado accediendo a la clave `salida` del resultado de la ejecución del flujo de trabajo.

In [66]:
resultado['salida']

'Autenticación exitosa. Bienvenido!'

### *5.5. Cómo Funcionan Juntos Estos Componentes*

LangGraph combina estos componentes en un marco coherente:
1. Los **nodos** realizan acciones según su funcionalidad definida.

2. Los **estados** contienen datos que los nodos utilizan y actualizan.

3. Las **aristas** garantizan una ejecución fluida al conectar los nodos en un orden lógico.

4. Las **aristas condicionales** añaden inteligencia al permitir la toma de decisiones y flujos de trabajo dinámicos.

## 6. Creación de un Flujo de Trabajo de Control de Calidad Específico para el Proyecto Guiado

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BgP-ruk_KS5H8D7iISsJ6A/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM.png" alt="Screenshot" width="150">
</div>

Actualmente, estamos diseñando un **flujo de trabajo de preguntas y respuestas (QA)** específicamente adaptado a un proyecto guiado. Este flujo de trabajo aprovecha LangGraph para crear transiciones modulares basadas en estados y garantiza que las preguntas relacionadas con el proyecto guiado se prioricen y se gestionen eficazmente.

El flujo de trabajo evalúa si la consulta del usuario es relevante para el proyecto guiado (por ejemplo, ¿de qué trata este proyecto guiado? o ¿qué es LangGraph?). Para las preguntas relevantes, utiliza un contexto predefinido para generar una respuesta informada. Si la consulta no está relacionada con el proyecto guiado, el flujo de trabajo comunica explícitamente que no hay suficiente contexto para proporcionar una respuesta, garantizando así la claridad y la transparencia en las interacciones.

### *6.1. Descripción del flujo de trabajo*

1. **Nodo de validación de entrada**

- **Propósito**: Asegura que el usuario haya ingresado una pregunta válida.

- **Flujo**: Si la entrada es válida, procede a evaluar la relevancia de la consulta; de lo contrario, finaliza con un mensaje de error.

2. **Nodo proveedor de contexto**

- **Propósito**: Verifica si la pregunta es específica del proyecto guiado.

- Para preguntas relevantes, proporciona contexto predefinido específico del proyecto.

- Para preguntas no relacionadas, establece el contexto en `null`.

- **Flujo**: Siempre pasa al paso de respuesta a la pregunta, independientemente de si el contexto está disponible o no.

3. **Nodo de respuesta a preguntas de LLM**

- **Propósito**: Utiliza el contexto (si está disponible) para responder la pregunta.

- Si se proporciona contexto, genera una respuesta detallada. 
- Si el contexto es `null`, responde con: *"No tengo suficiente contexto para responder a tu pregunta. Por favor, pregunta sobre el proyecto guiado."*

### *6.2. Definición del Estado Para el Flujo de Trabajo de Preguntas y Respuestas*

In [67]:
# Define el estado para el flujo de trabajo de preguntas y respuestas (QA) específico para el proyecto guiado.
class EstadoQA(TypedDict):
    # 'pregunta' Almacena la pregunta introducida por el usuario. Puede ser una cadena de texto o None si no se proporciona.
    pregunta: Optional[str]
    
    # 'contexto' Almacena el contexto relevante sobre el proyecto guiado, si la pregunta se refiere a él.
    # Si la pregunta no está relacionada con el proyecto, este valor será None.
    contexto: Optional[str]
    
    # 'respuesta' Almacena la respuesta generada. Puede ser None hasta que se genere la respuesta.
    respuesta: Optional[str]

Para devolver un objeto de ejemplo:

In [70]:
# Crear un objeto de ejemplo para el estado de QA.
ejemplo_estado_qa = EstadoQA(
    pregunta= "¿Cual es el propósito de este proyecto guiado?",
    contexto= "Este proyecto se centra en construir un chatbot utilizando Python.",
    respuesta= None
)

# Imprimir los atributos del objeto de ejemplo para verificar su contenido.
for clave, valor in ejemplo_estado_qa.items():
    print(f"{clave}: {valor}")

pregunta: ¿Cual es el propósito de este proyecto guiado?
contexto: Este proyecto se centra en construir un chatbot utilizando Python.
respuesta: None


#### 6.2.1. Definición del nodo de validación de entrada


<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QNWh9LRDo4A3uF5cz4bXcQ/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh.png" alt="Screenshot" width="150">
</div>

En este nodo, validamos la entrada del usuario (la pregunta). El nodo comprueba si se ha proporcionado una pregunta y si no está vacía. Si la pregunta está vacía, devuelve un mensaje de error que indica que no puede estar vacía. Si la pregunta es válida, se pasa al siguiente nodo.

In [71]:
def nodo_entrada_validacion(estado):
    # Extraer la pregunta del estado, y eliminar cualquier espacio en blanco al principio o al final.
    pregunta = estado.get("pregunta", "").strip()
    
    # Si la pregunta está vacía, devolver un mensaje de error indicando que la entrada no es válida.
    if not pregunta:
        return {"valido": False, "error": "La pregunta no puede estar vacía."}
    
    # Si la pregunta es válida, devolver un estado indicando que es válido.
    return {"valido": True}

Si la pregunta es válida, devuelve un estado válido:

In [72]:
nodo_entrada_validacion(ejemplo_estado_qa)

{'valido': True}

#### 6.2.2. Definición del proveedor de contexto

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/htDP3RKH9b6X1NHSKME-YQ/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-1-.png" alt="Screenshot" width="150">
</div>

Este nodo comprueba si la pregunta está relacionada con el proyecto guiado. Si menciona "LangGraph" o "proyecto guiado", proporciona el contexto pertinente. De lo contrario, establece el contexto en `None`.

In [74]:
def nodo_proveedor_contexto(estado):
    pregunta = estado.get("pregunta", "").lower()

    # Revisar si la pregunta está relacionada con el proyecto guiado. Si menciona "LangGraph" o "proyecto guiado", proporcionar el contexto relevante.
    if "langgraph" in pregunta or "proyecto guiado" in pregunta:
        contexto = (
            "Este proyecto guiado es sobre el uso de LangGraph, una biblioteca de Python para diseñar flujos de trabajo basados en estado."
            "LangGraph simplifica la construcción de aplicaciones complejas conectando nodos modulares con bordes condicionales."
        )
        return {"contexto": contexto}
    
    # Si la pregunta no está relacionada con el proyecto guiado, establecer el contexto como None.
    return {"contexto": None}

#### 6.2.3. Integración de LLM para el flujo de trabajo de control de calidad

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Djs-AScfvwE8fnKButiPXg/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-2-.png" alt="Screenshot" width="150">
</div>


En este paso, creamos un nodo que utiliza un LLM (Modelo de Lenguaje a Gran Escala) para responder a las preguntas del usuario según el contexto proporcionado. Si la pregunta no está relacionada con el proyecto guiado, el nodo la gestiona adecuadamente devolviendo una respuesta predefinida. Este enfoque utiliza la interfaz del modelo `meta-llama/llama-4-maverick-17b-128e-instruct-fp8`.

In [75]:
def nodo_llm_qa(estado):
    # Extraer la pregunta y el contexto del estado.
    pregunta = estado.get("pregunta", "")
    contexto = estado.get("contexto", None)

    # Verificar si el contexto es suficiente para responder a la pregunta. Si no hay contexto, devolver una respuesta indicando que no se puede responder a la pregunta.
    if not contexto:
        return {"respuesta": "No tengo suficiente contexto para responder a su pregunta."}

    # Construir el prompt dinámicamente utilizando la pregunta y el contexto. El prompt instruye al modelo a responder la pregunta basándose en el contexto proporcionado.
    prompt = f"Contexto: {contexto}\nPregunta: {pregunta}\nResponde la pregunta basándote en el contexto proporcionado."

    # Uso el modelo de lenguaje para obtener la respuesta.
    try:
        respuesta = llm_openai.invoke(prompt)
        return {"respuesta": respuesta.content.strip()}
    
    except Exception as e:
        return {"respuesta": f"An error occurred: {str(e)}"}

#### 6.2.4. Creación del grafo del flujo de trabajo de control de calidad

In [76]:
flujo_trabajo_qa = StateGraph(EstadoQA)

Ahora, agregamos el **`nodo-entrada`** al flujo de trabajo de control de calidad. Este nodo se encarga de validar si la pregunta proporcionada por el usuario no está vacía y tiene el formato correcto.

In [77]:
flujo_trabajo_qa.add_node("nodo-entrada", nodo_entrada_validacion)

Ahora, añadimos el **`nodo-contexto`** al flujo de trabajo de control de calidad. Este nodo determina si la pregunta está relacionada con el proyecto guiado y proporciona el contexto pertinente.

In [78]:
flujo_trabajo_qa.add_node("nodo-contexto", nodo_proveedor_contexto)

Ahora, agregamos el **`nodo-qa`** al flujo de trabajo de control de calidad. Este nodo utiliza un LLM para responder preguntas basadas en el contexto proporcionado.

In [79]:
flujo_trabajo_qa.add_node("nodo-qa", nodo_llm_qa)

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/0K3mU7FSZFw0tnAXiN22ag/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-3-.png" alt="Screenshot" width="150">
</div>

Ahora, establecemos el **punto de entrada** de nuestro flujo de trabajo de control de calidad en el **`nodo-entrada`**. Este es el primer nodo que se ejecutará al iniciar el flujo de trabajo, lo que garantiza que la entrada del usuario se valide antes de que se realicen otras operaciones.

In [80]:
flujo_trabajo_qa.set_entry_point("nodo-entrada")

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VFdsfj9Q26M9z45QilEVrQ/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-4-.png" alt="Screenshot" width="150">
</div>

En este paso, definimos la conexión o **arista** entre el **`nodo-entrada`** y el **`nodo-contexto`**. Esto significa que, una vez que el **`nodo-entrada`** valida la entrada del usuario, el flujo de trabajo pasará automáticamente al **`nodo-contexto`** para determinar si se necesita contexto adicional para la pregunta.

In [81]:
flujo_trabajo_qa.add_edge("nodo-entrada", "nodo-contexto")

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/kd4OXfGZ4oQeTipTjIq6GQ/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-5-.png" alt="Screenshot" width="150">
</div>

Ahora, definimos la conexión o **arista** entre el **`nodo-contexto`** y el **`nodo-qa`**. Una vez que el **`nodo-contexto`** haya determinado si hay contexto disponible para la pregunta, el flujo de trabajo pasará al **`nodo-qa`** para proporcionar la respuesta en función de la pregunta y el contexto.

In [82]:
flujo_trabajo_qa.add_edge("nodo-contexto", "nodo-qa")

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZBynpO0vyZbUD3fzWVc6LQ/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-6-.png" alt="Screenshot" width="150">
</div>

Ahora, definimos el extremo final de nuestro flujo de trabajo. Después de que el **`nodo-qa`** haya generado una respuesta basada en el contexto y la pregunta del usuario, pasamos al nodo **`END`**, que representa la finalización del flujo de trabajo.

In [83]:
flujo_trabajo_qa.add_edge("nodo-qa", END)

Ahora que hemos añadido los nodos y las aristas, podemos compilar el flujo de trabajo mediante el método `compile()`. Esto convierte el flujo de trabajo en una aplicación ejecutable que procesa los datos de entrada y sigue el flujo definido.

In [84]:
app_qa = flujo_trabajo_qa.compile()

#### 6.2.5. Hacer una pregunta irrelevante

Ahora, podemos activar el flujo de trabajo de control de calidad compilado introduciendo una pregunta irrelevante. El sistema procesará esta pregunta y avanzará por los nodos. Dado que la pregunta no guarda relación con el proyecto guiado, el sistema proporcionará una respuesta indicando que no dispone de suficiente contexto.

In [86]:
app_qa.invoke({"pregunta": "¿Cual es clima hoy?"})

{'pregunta': '¿Cual es clima hoy?',
 'contexto': None,
 'respuesta': 'No tengo suficiente contexto para responder a su pregunta.'}

#### 6.2.6. Formular una pregunta relevante

A continuación, formulamos una pregunta relevante relacionada con el proyecto guiado. El sistema utilizará el contexto proporcionado sobre LangGraph y responderá con una respuesta basada en la información pertinente.

In [88]:
app_qa.invoke({"pregunta": "Que es LangGraph?"})

{'pregunta': 'Que es LangGraph?',
 'contexto': 'Este proyecto guiado es sobre el uso de LangGraph, una biblioteca de Python para diseñar flujos de trabajo basados en estado.LangGraph simplifica la construcción de aplicaciones complejas conectando nodos modulares con bordes condicionales.',
 'respuesta': 'LangGraph es una biblioteca de Python que permite diseñar flujos de trabajo basados en estado, facilitando la construcción de aplicaciones complejas. Esta herramienta conecta nodos modulares mediante bordes condicionales, lo que simplifica el proceso de creación de estos flujos de trabajo.'}

In [89]:
app_qa.invoke({"pregunta": "Cual es el propósito de este proyecto guiado?"})

{'pregunta': 'Cual es el propósito de este proyecto guiado?',
 'contexto': 'Este proyecto guiado es sobre el uso de LangGraph, una biblioteca de Python para diseñar flujos de trabajo basados en estado.LangGraph simplifica la construcción de aplicaciones complejas conectando nodos modulares con bordes condicionales.',
 'respuesta': 'El propósito de este proyecto guiado es enseñar a utilizar LangGraph, una biblioteca de Python, para diseñar flujos de trabajo basados en estado. A través de este proyecto, se busca simplificar la construcción de aplicaciones complejas mediante la conexión de nodos modulares con bordes condicionales, facilitando así el desarrollo de aplicaciones más estructuradas y eficientes.'}

## 7. Ejercicios

En este ejercicio, crearás un contador simple usando LangGraph.

### *7.1. Ejercicio 1 - Definir el Tipo de Estado*

Aquí se definirá el esquema de estado utilizado por el grafo. Este esquema deberá registrar lo siguiente:
- `n`: un contador que comienza en 1.

- `letra`: una letra minúscula generada aleatoriamente en cada paso.

In [90]:
import random
import string
from typing import TypedDict

from langgraph.graph import StateGraph, END

In [91]:
class EstadoContador(TypedDict):
    n: int
    letra: str

### *7.2. Ejercicio 2 - Crear la Función `agregar()`*

Este nodo debe representar el paso de `incremento` de la siguiente manera:
- Suma 1 al valor actual de n.

- Selecciona aleatoriamente una letra minúscula y actualiza el campo correspondiente.

In [92]:
def agregar(estado):
    n = estado.get("n", 0) + 1    
    letra = random.choice(string.ascii_lowercase)
    
    return {"n": n, "letra": letra}

### *7.3. Ejercicio 3 - Crear la Función `imprimir_salida()`*

In [93]:
def imprimir_salida(estado):
    n = estado.get("n", 0)
    letra = estado.get("letra", "")
    print(f"Contador: {n}, Letra: {letra}")
    
    return estado

### *7.4. Ejercicio 4 - Condición de Parada*

Crea una función con una condición de terminación:
- Si el contador alcanza 13 o más, el flujo de trabajo debe finalizar.

- De lo contrario, debe volver a agregar un nodo.

In [94]:
def decision(estado):
    n = estado.get("n", 0)
    
    if n >= 13:
        return "terminar"
    else:
        return "continuar"

### *7.5. Ejercicio 5 - Construcción de Grafos*

En este ejercicio, construirás el flujo de LangGraph:

- Crea un objeto `StateGraph` usando el `EstadoContador` que creaste.

- Agrega los nodos `agregar` y `imprimir_salida`.

- Agrega una arista entre `agregar` y `imprimir_salida`.
- Agrega una arista condicional entre `imprimir_salida` y `END` según la condición `decision`.

- Establece `agregar` como punto de entrada del grafo.

In [96]:
flujo_trabajo_contador = StateGraph(EstadoContador)

flujo_trabajo_contador.add_node("agregar", agregar)
flujo_trabajo_contador.add_node("imprimir_salida", imprimir_salida)


flujo_trabajo_contador.set_entry_point("agregar")
flujo_trabajo_contador.add_edge("agregar", "imprimir_salida")

flujo_trabajo_contador.add_conditional_edges("imprimir_salida", decision, {"terminar": END, "continuar": "agregar"})

app_contador = flujo_trabajo_contador.compile()

### *Ejercicio 6 - Compilar y Ejecutar*

Compila el gráfico e inicia la ejecución con la entrada inicial proporcionada:
- El contador debe comenzar en 1.

- Deja la letra vacía (se completará con el nodo de suma).

In [97]:
entrada = {
    "n": 0
}

app_contador.invoke(entrada)

Contador: 1, Letra: r
Contador: 2, Letra: o
Contador: 3, Letra: b
Contador: 4, Letra: p
Contador: 5, Letra: b
Contador: 6, Letra: q
Contador: 7, Letra: s
Contador: 8, Letra: p
Contador: 9, Letra: c
Contador: 10, Letra: w
Contador: 11, Letra: j
Contador: 12, Letra: b
Contador: 13, Letra: b


{'n': 13, 'letra': 'b'}

Copyright © 2024 IBM Corporation. All rights reserved.
